In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import math

In [3]:
sign_image_path = "./Task-1/rh_sign.jpg"
scene_image_path = "./Task-1/img1.png"

# Load images
sign_image = cv2.imread(sign_image_path, cv2.IMREAD_COLOR)
scene_image = cv2.imread(scene_image_path, cv2.IMREAD_COLOR)

In [3]:
def bgr_to_rgb(image):
   return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

In [4]:
def hsv_to_rgb(image):
   return cv2.cvtColor(image, cv2.COLOR_HSV2RGB)

In [5]:

def plot_images(images, titles=None, cols=3, figsize=(15, 10)):
    # Number of images
    num_images = len(images)
    
    # Calculate number of rows
    rows = math.ceil(num_images / cols)
    
    # Create the subplot grid
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.ravel()  # Flatten the 2D grid of axes to iterate over
    
    # Loop through each image and corresponding axis
    for i in range(rows * cols):
        if i < num_images:
            # Display image
            axes[i].imshow(images[i], cmap='gray' if len(images[i].shape) == 2 else None)
            
            # Add title if provided
            if titles:
                axes[i].set_title(titles[i], fontsize=12)
        else:
            # Hide any unused subplot
            axes[i].axis('off')
        
        # Remove axes for clarity
        axes[i].axis('off')
    
    # Adjust layout for better spacing
    plt.tight_layout()
    plt.show()

In [6]:
def boost_reds(image):
    # Convert to HSV
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Split HSV channels
    h, s, v = cv2.split(hsv)

    # Target red hues (0-10 and 160-180 degrees in HSV)
    mask1 = cv2.inRange(h, 0, 10)
    mask2 = cv2.inRange(h, 160, 180)
    red_mask = cv2.bitwise_or(mask1, mask2)

    # Boost saturation for red hues
    s = cv2.addWeighted(s, 1.0, red_mask, 0.1, 0)  # Add 10% of the red mask to saturation

    # Merge channels and convert back to BGR
    boosted_hsv = cv2.merge((h, s, v))
    boosted_image = cv2.cvtColor(boosted_hsv, cv2.COLOR_HSV2BGR)

    return boosted_image


In [7]:
def find_reds(img):
    img_hsv=cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # lower mask (0-10)
    lower_red = np.array([0,50,50])
    upper_red = np.array([10,255,255])
    mask0 = cv2.inRange(img_hsv, lower_red, upper_red)

    # upper mask (170-180)
    lower_red = np.array([170,50,50])
    upper_red = np.array([180,255,255])
    mask1 = cv2.inRange(img_hsv, lower_red, upper_red)

    # join my masks
    mask = mask0+mask1

    # set my output img to zero everywhere except my mask
    output_img = img.copy()
    output_img[np.where(mask==0)] = 0

    # or your HSV image, which I *believe* is what you want
    # output_hsv = img_hsv.copy()
    return output_img

In [8]:
def boost_contrast(image):
    # # Create a CLAHE object with specified clip limit and tile grid size
    # clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    # # Convert the image to LAB color space (to apply CLAHE only to the L channel)
    # scene_image_lab = cv2.cvtColor(scene_image, cv2.COLOR_BGR2LAB)

    # # Split the LAB image into L, A, and B channels
    # l, a, b = cv2.split(scene_image_lab)

    # # Apply CLAHE to the L channel (lightness channel)
    # l_clahe = clahe.apply(l)

    # # Merge the CLAHE enhanced L channel back with the A and B channels
    # scene_image_lab_clahe = cv2.merge((l_clahe, a, b))

    # # Convert the image back to BGR color space
    # scene_image = cv2.cvtColor(scene_image_lab_clahe, cv2.COLOR_LAB2BGR)
    # scene_image = cv2.cvtColor(scene_image, cv2.COLOR_BGR2HSV)
    # sign_image = cv2.cvtColor(sign_image, cv2.COLOR_BGR2HSV)
    pass

In [9]:
def find_red_contour(image):
    original = image.copy()
    # Convert to HSV color space
    print("got here")
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Define red color range in HSV
    lower_red1 = np.array([0, 70, 50])
    upper_red1 = np.array([10, 255, 255])
    lower_red2 = np.array([170, 70, 50])
    upper_red2 = np.array([180, 255, 255])

    # Create masks for red
    mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
    mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
    red_mask = mask1 + mask2

    # Find contours
    contours, _ = cv2.findContours(red_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for contour in contours:
        # Approximate the shape
        epsilon = 0.04 * cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, epsilon, True)

        # Check if the shape is a triangle (3 vertices)
        if len(approx) == 3:
            # Draw the contour and mark the triangle
            cv2.drawContours(original, [contour], -1, (0, 255, 0), 2)
            cv2.putText(original, "Triangle", tuple(approx[0][0]), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Show the result
    cv2.imshow("Detected Triangle", original)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [10]:
def locate_rh_sign(sign_image, scene_image):
    plot_images([bgr_to_rgb(sign_image), find_red_contour(boost_reds(scene_image)), bgr_to_rgb(scene_image), find_red_contour(scene_image)])
    return
    
    sign_image = find_reds(sign_image)
    scene_image = boost_reds(scene_image)

    # Create keypoints and descriptors using SIFT
    sift = cv2.SIFT_create()
    query_keypoints, query_descriptors = sift.detectAndCompute(sign_image, None)
    target_keypoints, target_descriptors = sift.detectAndCompute(scene_image, None)

    # Match features using BFMatcher
    bf = cv2.BFMatcher()
    matches = bf.match(query_descriptors, target_descriptors)
    
    # Sort matches by distance
    matches = sorted(matches, key=lambda x: x.distance)
    
    # Filter best 20% of matches
    good_matches = matches[:int(len(matches) * 0.25)]

    # Draw matches on the images
    match_image = cv2.drawMatches(sign_image, query_keypoints, scene_image, target_keypoints, good_matches, None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

    # Plot the image with matches
    plt.figure(figsize=(12, 6))
    plt.title("Keypoint Matches")
    plt.imshow(match_image)
    plt.show()


    # Extract locations of good matches
    src_pts = np.float32([query_keypoints[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([target_keypoints[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

    # Find homography matrix
    M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 3.0)
    matches_mask = mask.ravel().tolist()  # Filter matches based on inliers

    print(matches_mask)
    
    # Get bounding box for the sign image
    h, w = sign_image.shape[:2]
    pts = np.float32([[0, 0], [0, h], [w, h], [w, 0]]).reshape(-1, 1, 2)
    dst = None
    if M is not None:
        dst = cv2.perspectiveTransform(pts, M)
    
    # Draw bounding box on the scene image
    result_img = scene_image.copy()
    if dst is not None and sum(matches_mask) > 4:  # Ensure at least 4 inliers
        dst = np.int32(dst)
        cv2.polylines(result_img, [dst], True, (0, 255, 0), 3, cv2.LINE_AA)
    
    # Show result
    plt.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
    plt.title("Detected Road Sign")
    plt.show()
    
    # return result_img



In [ ]:
import cv2
import numpy as np

def preprocess_with_edges(image):
    """Preprocess the image using edge detection."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, 50, 150)  # Edge detection
    return edges

def is_triangle(contour):
    """Check if a contour is a triangle based on vertex count and other properties."""
    # Approximate the contour to a polygon
    epsilon = 0.04 * cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, epsilon, True)

    # Check if the shape has 3 vertices
    if len(approx) == 3:
        # Check aspect ratio (optional, adjust as needed)
        x, y, w, h = cv2.boundingRect(approx)
        aspect_ratio = w / h
        if 0.8 < aspect_ratio < 1.2:  # Close to equilateral triangle
            return True
    return False

# Load the reference and search images
reference_image = cv2.imread(sign_image_path)
search_image = cv2.imread(scene_image_path)


# Preprocess the images to get edges
reference_edges = preprocess_with_edges(reference_image)
search_edges = preprocess_with_edges(search_image)

# Find contours in the reference image
reference_contours, _ = cv2.findContours(reference_edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Filter out the triangle contour from the reference image
reference_triangle_contour = None
for contour in reference_contours:
    if is_triangle(contour):
        reference_triangle_contour = contour
        break  # We assume there's only one triangle

# If no triangle is found in the reference image, exit
if reference_triangle_contour is None:
    print("No triangle found in reference image.")
    exit()

# Find contours in the search image
search_contours, _ = cv2.findContours(search_edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Match the reference triangle with contours from the search image
best_match = None
best_score = float('inf')

for contour in search_contours:
    if is_triangle(contour):
        match_score = cv2.matchShapes(reference_triangle_contour, contour, cv2.CONTOURS_MATCH_I1, 0.0)
        if match_score < best_score:
            best_match = contour
            best_score = match_score

# If a match is found, draw it on the search image
if best_match is not None:
    cv2.drawContours(search_image, [best_match], -1, (0, 255, 0), 2)
    cv2.putText(search_image, f"Best Match: {best_score:.2f}", tuple(best_match[0][0]),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

# Show the result
cv2.imshow("Detected Triangle", search_image)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [ ]:
locate_rh_sign(sign_image, scene_image)

got here
